In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession. \
builder. \
appName("week5-assignment-2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:

data = [
    ("Alice", "Math", 90),
    ("Alice", "Science", 85),
    ("Alice", "Math", 95),
    ("Bob", "Math", 70),
    ("Bob", "Science", 75),
    ("Charlie", "Science", 88),
]
columns = ["student", "subject", "score"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()


Original DataFrame:
+-------+-------+-----+
|student|subject|score|
+-------+-------+-----+
|  Alice|   Math|   90|
|  Alice|Science|   85|
|  Alice|   Math|   95|
|    Bob|   Math|   70|
|    Bob|Science|   75|
|Charlie|Science|   88|
+-------+-------+-----+



In [3]:

# collect_list keeps duplicates and does not guarantee order
grouped = df.groupBy("student").agg(
    collect_list("subject").alias("subjects"),
    collect_list("score").alias("scores"),
)

print("Grouped with collect_list (duplicates kept):")
grouped.show(truncate=False)

# collect_set for comparison, dedupes the values
grouped_distinct = df.groupBy("student").agg(
    collect_set("subject").alias("distinct_subjects")
)

print("Grouped with collect_set (duplicates removed):")
grouped_distinct.show(truncate=False)

Grouped with collect_list (duplicates kept):
+-------+---------------------+------------+
|student|subjects             |scores      |
+-------+---------------------+------------+
|Charlie|[Science]            |[88]        |
|Bob    |[Math, Science]      |[70, 75]    |
|Alice  |[Math, Science, Math]|[90, 85, 95]|
+-------+---------------------+------------+

Grouped with collect_set (duplicates removed):
+-------+-----------------+
|student|distinct_subjects|
+-------+-----------------+
|Charlie|[Science]        |
|Bob    |[Math, Science]  |
|Alice  |[Math, Science]  |
+-------+-----------------+

